# 07 · Observability, Governance and Delivery

> **The story:** The team works. Now DLD's IT department asks: what did it do, what did it cost, who owns it, and how do staff use it?

**You will learn**
- An inventory of your agents and their identities
- End-to-end traces: every step every agent took
- Cost of your resource group
- Publishing to Teams / Microsoft 365 Copilot and governing agents with Agent 365
- Cleaning up so you stop paying

**Time:** about 15 minutes · **Before you start:**
- Notebooks 00 to 06 finished

<sub>Workshop material: the rental documents summarise Dubai rules for training only. Not legal advice.</sub>

In [ ]:
# @title Setup: run this first (about 1 minute, then sign in)
from pathlib import Path
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # The workshop code goes on this notebook's temporary Colab machine (a few seconds; nothing in Drive).
    ROOT = Path("/content/dld-foundry-workshop")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)   # pick up any fixes
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/NetwaysPresales/dld-foundry-workshop", str(ROOT)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in [here, *here.parents] if (p / "dld_workshop").exists()), here)
sys.path.insert(0, str(ROOT))

from azure.identity import AzureCliCredential, DeviceCodeCredential
from dld_workshop import WorkshopEnvironment, WorkshopSettings
from dld_workshop.display import show, table, checklist

# In Colab you'll see a link and a code: open the link, enter the code, sign in with your work account.
credential = DeviceCodeCredential() if IN_COLAB else AzureCliCredential()

settings = WorkshopEnvironment.discover(credential)
print(f"✅ Signed in. Your environment: {settings.resource_group}")

## Step 1 · Agent inventory

In [ ]:
from azure.ai.projects import AIProjectClient
from dld_workshop.agents import RULES_AGENT, MAIL_AGENT, CALENDAR_AGENT
from dld_workshop.hosting import HOSTED_AGENT

project = AIProjectClient(settings.project_endpoint, credential)
existing = {a.name for a in project.agents.list()}
inventory = []
for name in [RULES_AGENT, MAIL_AGENT, CALENDAR_AGENT, HOSTED_AGENT]:
    if name not in existing:
        inventory.append({"agent": name, "latest version": "-", "own identity": "not created"})
        continue
    details = project.agents.get(name)
    inventory.append({
        "agent": name,
        "latest version": str(details.versions.latest.version),
        "own identity": details.instance_identity.principal_id if details.instance_identity else "-",
    })
table(inventory)

## Step 2 · Traces

Every step is recorded in Application Insights: manager → agent → tool → model. Traces appear 2 to 5 minutes after a run.

In [ ]:
from dld_workshop.observability import query_logs, TRACE_SUMMARY, AGENT_SPANS

table(query_logs(settings, credential, TRACE_SUMMARY))
table(query_logs(settings, credential, AGENT_SPANS)[:20])

## Step 3 · Cost

Azure cost data lags usage by several hours, so today's run usually shows up tomorrow. You can also open **Cost analysis** on the resource group.

In [ ]:
from dld_workshop.observability import resource_group_cost

cost = resource_group_cost(settings, credential)
if cost.get("available") and cost["rows"]:
    table(cost["rows"])
else:
    print("No cost rows yet. Check again later.")

## Step 4 · Reaching staff and governing agents (portal walkthrough)
The facilitator demonstrates these in the portal.

1. **Tracing** in your Foundry project: open one mode 1 run and follow manager → Mail agent → Rules agent → Calendar agent.
2. **Agent identity** → Microsoft Entra: each agent's narrow role assignments.
3. **Evaluations**: compare versions before promoting one.
4. **Publish** the Rules agent to **Teams and Microsoft 365 Copilot** (scope: *Just you*). *Requires a Microsoft 365 Copilot licence.*
5. **Agent 365**: find the published agent in the registry: owner, permissions, Defender and Purview signals. *Requires admin enablement.*
6. **Copilot Studio**: when business teams need low-code channels, connectors or flows on top of these agents.

## Step 5 · Clean up

Deletes your resource group permanently, including the Logic App, so nothing keeps running. Delete any test emails and calendar events yourself in Outlook. Type your exact resource-group name to confirm.

In [ ]:
from dld_workshop import WorkshopEnvironment

CONFIRM_DELETE = ""   # ← type your resource-group name here, e.g. "rg-dld-foundry-yourname01"
if CONFIRM_DELETE:
    WorkshopEnvironment(credential, settings).delete(CONFIRM_DELETE)
    print("✅ Everything deleted. You are no longer being charged.")
else:
    print(f"⚠️ {settings.resource_group} still exists and is billable. Set CONFIRM_DELETE to clean up.")

> ### 👉 Follow along in the portal
>
> **Build → Tools**: delete the `WorkIQMail` and `WorkIQCalendar` connections if you created them in a project you keep.

---
## What to notice
- Every agent is inventoried with its own identity, traced, evaluated and costed.
- Publishing and fleet governance (Agent 365) sit on top of what you built; they don't replace it.

**Try it yourself:** in the Foundry portal, open **Tracing** and find the step where the manager called the Calendar agent.